# Group Attention / Grouped Query Attention

Group Attention 这里更准确地说，通常指 Grouped Query Attention（GQA，分组查询注意力）。它是 Multi-Head Attention（MHA）和 Multi-Query Attention（MQA）之间的一种折中形式，核心思想是：多个 Query heads 共享同一组 Key / Value heads。

## 从 MHA 到 GQA

在标准 Multi-Head Attention 中，如果有 $h$ 个 head，那么每个 head 都有自己的 Q、K、V：

$$
Q_i, K_i, V_i \quad i = 1,2,\cdots,h
$$

每个 head 独立计算：

$$
head_i = \operatorname{Attention}(Q_i, K_i, V_i)
$$

这种方式表达能力强，但在大语言模型推理时，K/V cache 的开销也会随着 head 数量一起增长。

在 Grouped Query Attention 中，Query 的 head 数量仍然较多，但 Key / Value 的 head 数量更少。设：

$$
h_q = \text{number of query heads}
$$

$$
h_{kv} = \text{number of key/value heads}
$$

通常有：

$$
h_q > h_{kv}
$$

并且要求：

$$
h_q \bmod h_{kv} = 0
$$

每组 query heads 共享同一个 K/V head。每组包含的 query head 数量为：

$$
g = \frac{h_q}{h_{kv}}
$$

对于第 $i$ 个 query head，它对应的 K/V 组可以写成：

$$
group(i) = \left\lfloor \frac{i}{g} \right\rfloor
$$

于是第 $i$ 个 head 的注意力计算为：

$$
head_i = \operatorname{Attention}(Q_i, K_{group(i)}, V_{group(i)})
$$

## Shape 理解

假设输入为：

$$
X \in \mathbb{R}^{B \times T \times d_{model}}
$$

Query heads 的 shape 为：

$$
Q: [B, h_q, T, d_{head}]
$$

Key / Value heads 的 shape 为：

$$
K,V: [B, h_{kv}, T, d_{head}]
$$

为了和每个 query head 计算 attention，K/V 通常会在 head 维度上 repeat 或 broadcast：

$$
K,V: [B, h_{kv}, T, d_{head}] \rightarrow [B, h_q, T, d_{head}]
$$

但需要注意，逻辑上 repeat 之后每一组 query heads 仍然共享同一份 K/V。这样可以减少 K/V 投影参数量，也可以在推理时显著减少 KV cache 的存储开销。

## MHA、GQA、MQA 的关系

可以用 $h_q$ 和 $h_{kv}$ 的关系来理解三者：

| 类型 | Query heads | Key/Value heads | 特点 |
| ---- | ---- | ---- | ---- |
| MHA | $h_q$ | $h_{kv}=h_q$ | 每个 query head 都有独立 K/V，表达能力强，KV cache 开销大 |
| GQA | $h_q$ | $1 < h_{kv} < h_q$ | 多个 query heads 共享一组 K/V，在效果和效率之间折中 |
| MQA | $h_q$ | $h_{kv}=1$ | 所有 query heads 共享同一组 K/V，KV cache 最省 |

## 为什么要用 GQA

在训练和推理大模型时，attention 的计算和缓存成本都很高。尤其在自回归生成中，每生成一个新 token，都需要保存历史 token 的 K/V，这就是 KV cache。

标准 MHA 中，每一层、每一个 head 都要保存自己的 K/V：

$$
KV\ Cache \propto h_q \times T \times d_{head}
$$

而 GQA 只需要保存较少的 K/V heads：

$$
KV\ Cache \propto h_{kv} \times T \times d_{head}
$$

因为 $h_{kv} < h_q$，所以 GQA 可以降低推理时的显存占用和带宽压力，同时保留比 MQA 更强的表达能力。

## 和当前 MultiHeadAttention 实现的关系

当前 `MultiHeadAttention.ipynb` 中实现的是标准 MHA，也就是：

$$
h_{kv} = h_q
$$

如果要继续实现 GQA，关键改动是：

- Q 仍然投影成 `num_query_heads` 个 head。
- K/V 只投影成 `num_kv_heads` 个 head。
- 计算 attention 前，把 K/V 在 head 维度 repeat 到和 Q 相同的 head 数量。
- mask 的 shape 仍然可以广播到 `[B, h_q, T, T]`。

因此，GQA 可以看作是在 Multi-Head Attention 的基础上，对 K/V heads 做了分组共享。
